# Cloud Data Integration - Hands-on Lab #

Denodo Platform (Virtual DataPort) implements the **Arrow Flight SQL Protocol**, allowing connections from clients interacting with this protocol independently of its underlying programming language. 

In this Hands-on lab we will connect to Denodo PLatform from a native Python application, using the Flight SQL Python driver. This driver implements the specification "ADBC: Arrow Database Connectivity", a set of APIs for accessing to Arrow-native databases.

We will start with the installation of the Python packages.

In [ ]:
pip install adbc_driver_flightsql

### Connect with a Denodo database using Arrow Flight SQL ###

We will use the same Denodo credentials (the user and password used previously for connecting to the Design Studio and Data Marketplace)

In [ ]:
from adbc_driver_flightsql.dbapi import connect
from IPython.display import display
      
conn=connect(
        f"grpc://vdp:9994",
        db_kwargs={
            "username": "denodo",
            "password": "denodo",
            "adbc.flight.sql.rpc.call_header.database": "denodo_hol",
            "adbc.flight.sql.rpc.call_header.timePrecision": "milliseconds",
        },
        autocommit=True
    )

with conn.cursor() as cur:
    cur.execute(f"""
        select c_customer_sk, total_sales from denodo_hol.sales_by_customer limit 5' 
        """)
    result = cur.fetch_df()
    display(result)

#### Using other python libraries (Pandas) ####
Now, we will create a Pandas DataFrame loading the complete data set of our "sales_by_customer" view into the DataFrame

In [ ]:
with conn.cursor() as cur:
    cur.execute(f"""
        select * from denodo_hol.sales_by_customer' 
        """)
    result = cur.fetchallarrow()
    df = result.to_pandas()

Let's introspect the results of the DataFrame

In [ ]:
df.head()

#### Printing the output in a Bar Chart ####

In [ ]:
import matplotlib.pyplot as plt

#create the buckets
bins = [100,200,300,400,500,600,700,800,900,1000,1100]

#plot the total sales for each bucket
plt.hist(df.total_sales, bins, histtype='bar')
plt.xlabel('total sales in USD') 
plt.ylabel('number of customers')